# Session 0a - Environment Setup

**Asynchronous · ~45 minutes · do this first**

---

## What you are doing and why

You will install a **pinned** environment: exact versions of every library, chosen
once and identical for everyone on the course.

That is not fussiness. Between two scikit-learn releases the way cross-validation
assigns rows to folds changed enough to move this course's headline numbers by 0.045 -
same data, same code, different answer. Every figure in these notebooks was
measured against the versions below. If your versions differ, your numbers will
differ, and you will spend an evening wondering why.

## Checklist

1. Python 3.12 installed
2. A virtual environment created **inside the course folder**
3. `pip install -r requirements.txt`
4. `python tools/check_setup.py` exits cleanly
5. JupyterLab opens and can run a cell
6. You can find `data/raw/listings_barcelona_2026-06-24.csv.gz`

## §1 - Python 3.12

Check what you have:

```bash
python --version
```

You want `Python 3.12.x`. If you have 3.11 or 3.13 the course will *probably* work,
but you are off the tested path - install 3.12 from
[python.org/downloads](https://www.python.org/downloads/) or via your package manager.

If `python` is not found but `python3` is, use `python3` everywhere below.

## §2 - A virtual environment

A virtual environment is a private copy of Python for this course, so the versions we
pin cannot break anything else on your machine.

From inside the course folder:

```bash
python -m venv .venv
```

Then **activate** it - this is the step people forget:

| | command |
|---|---|
| Windows (PowerShell) | `.venv\Scripts\Activate.ps1` |
| Windows (cmd) | `.venv\Scripts\activate.bat` |
| macOS / Linux | `source .venv/bin/activate` |

Your prompt should now start with `(.venv)`. **Confirm you are in it:**

```bash
where python      # Windows - should point inside .venv
which python      # macOS / Linux - same
```

If that path does not contain `.venv`, activation did not work, and everything you
install next will go somewhere else. Stop and fix it before continuing.

> **On PowerShell execution policy.** If activation fails with "running scripts is
> disabled", run once:
> `Set-ExecutionPolicy -Scope CurrentUser -ExecutionPolicy RemoteSigned`

## §3 - Install

```bash
pip install -r requirements.txt
```

About 200 MB. Do **not** install `requirements-dl.txt` yet - that is PyTorch, and it
is needed from Session 8, not Session 1.

## §4 - Verify

```bash
python tools/check_setup.py
```

This checks every pinned version and loads the data snapshot. It should end with
`Setup is good. You are ready for Session 1.` If it does not, it tells you exactly
what to fix.

Run the same check from inside a notebook, which additionally proves your Jupyter
kernel is the venv and not some other Python:

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

print(f"python executable : {sys.executable}")
print(f"course root       : {ROOT}")
print()
print("If the executable path above does NOT contain '.venv', your notebook is running")
print("the wrong Python. Fix that before going further - see section 6.")

In [ ]:
import numpy as np
import pandas as pd
import sklearn

from src.data import PATHS, describe_environment, load_raw, set_seed

print(describe_environment().to_string(index=False))
print()
print("expected: python 3.12.x | numpy 2.5.2 | pandas 3.0.5 | sklearn 1.9.0")

## §5 - The data

The dataset ships with the repository. You should **not** download anything.

In [ ]:
print(f"looking for: {PATHS.snapshot.name}")
print(f"exists     : {PATHS.snapshot.exists()}")
if PATHS.snapshot.exists():
    size_mb = PATHS.snapshot.stat().st_size / 1e6
    print(f"size       : {size_mb:.1f} MB")
    set_seed()
    df = load_raw()
    print(f"loaded     : {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"expected   : 15,293 rows x 90 columns")
else:
    print("\nMISSING. Re-clone the repository or ask for the file.")
    print("Do NOT download a fresh snapshot from Inside Airbnb - snapshots rotate")
    print("quarterly and every figure in the course notes is tied to this exact file.")

## §6 - JupyterLab, and the kernel trap

```bash
jupyter lab
```

A browser tab opens. Navigate to `00_Prerequisites/` and open a notebook.

**The trap that catches almost everyone once.** JupyterLab can run notebooks against
a *different* Python than the one you activated. The symptom is a
`ModuleNotFoundError` for something you know you installed.

The fix is to register the venv as a named kernel:

```bash
python -m ipykernel install --user --name course --display-name "Python 3 (course)"
```

Then in JupyterLab pick **Kernel → Change Kernel → Python 3 (course)**. The §4 cell
above confirms you are on the right one.

### The four things worth knowing about notebooks

| | |
|---|---|
| `Shift+Enter` | run the cell and move on |
| `Esc` then `A` / `B` | insert a cell above / below |
| `Esc` then `M` / `Y` | make the cell markdown / code |
| Kernel → Restart & Run All | **the only honest test that your notebook works** |

That last one matters more than it sounds. A notebook is not a program; it is a
sequence of cells you may have run in any order. Cells can depend on variables that
no longer exist in the file, and a notebook that "works" in your session can fail
completely on a fresh kernel.

> **Before submitting any milestone: Restart & Run All.** If it does not run top to
> bottom, it does not run. This is graded - a number that cannot be reproduced by
> running your notebook caps the final report at a C.

## §7 - Reproducibility, from the start

Three habits, adopted now, that the course will keep asking for.

**1. Seed everything.** Every notebook's first code cell calls `set_seed(SEED)`.
Without it, your results change between runs and you cannot tell a real improvement
from noise.

**2. Record your versions.** `describe_environment()` above. Put it in anything you
submit. "It worked on my machine" is not a defence if nobody knows what your machine
had.

**3. Never edit `data/raw/`.** It is the single source of truth. Cleaned and derived
data goes in `data/interim/`. If you corrupt the raw file, every number you produce
afterwards is quietly wrong.

In [ ]:
# TODO: Confirm your setup is reproducible.
#
#   1. Call set_seed() and draw 5 random numbers with np.random.
#   2. Restart the kernel (Kernel -> Restart), re-run, and confirm you get the SAME
#      five numbers.
#   3. Now draw five WITHOUT seeding first, restart, and re-run. Different?
#
# Write one sentence on why step 3 would make a model comparison untrustworthy.

## §8 - When something breaks

In order. Most problems are one of the first three.

| symptom | almost certainly |
|---|---|
| `ModuleNotFoundError` for something you installed | notebook is on the wrong kernel - §6 |
| versions do not match the pins | installed outside the venv - check `where python` |
| `FileNotFoundError` on the data | running from the wrong directory, or an incomplete clone |
| `ImportError: cannot import name 'x' from 'src.data'` | course root not on `sys.path`; the first cell handles it |
| activation refused on PowerShell | execution policy - §2 |
| pip fails behind a corporate proxy | ask your IT team for the proxy settings; do not disable TLS verification |

If none of those: send me the **full** traceback, the output of
`python tools/check_setup.py`, and your operating system. A screenshot of the last
line of an error is not enough to diagnose anything - Session 0e's final section is
about reading tracebacks, and the same skill applies to reporting them.

## Next

1. `00b_Python_for_Data_Science` - skim if you are fluent, work through if not
2. `00c_NumPy_Essentials` - **required**, and the one that matters most
3. `00d_Visualization_Reference` - reference; read §1 and §7
4. `00e_Diagnostic_Quiz` - closed book, and send me the topic report

## Summary

- Pinned versions are not fussiness: a scikit-learn release moved this course's
  figures by 0.045.
- Activate the venv, and **verify with `where python`** before installing anything.
- Register the kernel, and check `sys.executable` from inside the notebook.
- **Restart & Run All** is the only honest test that a notebook works.
- Seed everything, record versions, never edit `data/raw/`.